In [3]:
import pandas as pd

In [4]:
df = pd.read_parquet("processing_data/metrics.parquet")

df

,constant_answer,KNN,LogisticRegression,RandomForestClassifier,MLPClassifier
accuracy,0.542947,0.658373,0.683057,0.692085,0.692133
precision,0.542947,0.670544,0.686860,0.691364,0.681096
recall,1.000000,0.728937,0.765034,0.781963,0.814191
f1_score,0.703779,0.698522,0.723843,0.733878,0.741720
roc_auc_score,0.500000,0.709543,0.739066,0.750338,0.748370


#### Выводы по задаче 2 (модель возврата, которую сможем применять на последних стадиях рек системы):

Во-первых, это новое экспериментальное решение, по нему у компании нет общепринятого стандарта, как такое оценивать перед выводом на АБ. Поэтому, в отличие от 1ой бизнес-задачи, тут мы просто делаем оценку train/test как хотим: не запариваемся с time-aware train/test split и просто оцениваем решение обычной задачи классификации.

Выбрали F1-score как целевую метрику, потому что от бизнеса звучало требование "хотим поменьше FP и поменьше FN". Остальные метрики просто как гардрейлы.

Мы не делали кроссвалидацию, но мы не боимся 1) того что оценка по тесту несостоятельная - 147к заказов в тесте, оценка хорошая 2) того что переобучимся - у нас вся логика поиска лучших моделей ложится на val выборку, по которой в процессе трейна мы gridsearch-ем ищем лучшие параметры.

Итак: мы видим, что у нас два фаворита - рандом форест и нейронка. По F1 немного лучше нейронка (MLPClassifier), её и выберем победителем. В данном случае нейронка не окажет доп. затрат на инфраструктуру, т.к. это простейший MLPClassifier (лучшая комбинация - 4 слоя ширины 25).

#### Из интересного: предикт самым частым классом пробивает KNN даже по такой умной метрике, как F1.

Да, понятно, что recall очевидно будет 100%, но константный предиктор выдаёт достаточно высокий precision. Чтобы бороться с такими странными эффектами, можно аккуратнее формулировать "относительно какого класса считать метрики".